# ==============================================================================
# 🚗 AUTOMOTIVE VALUATION: VEHICLE PRICING VIA SUPPORT VECTOR REGRESSOR
# ==============================================================================
### Core Principles:
Estimating vehicle market price from engine architecture (`enginesize`, `horsepower`, `curbweight`, `wheelbase`) and body design (`carbody`, `drivewheel`, `fueltype`).

Demonstrates combining `OneHotEncoder` and `StandardScaler` to train an SVR pipeline with an $\varepsilon$-tube that absorbs standard dealership pricing variance.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Automotive Valuation Environment ready.")


✅ STEP 1: Automotive Valuation Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Car Price Assignment dataset (205 vehicles).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/car_price/CarPrice_Assignment.csv'
df = pd.read_csv(data_path)

# Drop raw non-predictive IDs
df = df.drop(columns=['car_ID', 'CarName'])

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} vehicles, {df.shape[1]} attributes")
print(f"Vehicle Price Summary:\n{df['price'].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 205 vehicles, 24 attributes
Vehicle Price Summary:
count      205.00
mean     13276.71
std       7988.85
min       5118.00
25%       7788.00
50%      10295.00
75%      16503.00
max      45400.00
Name: price, dtype: float64


,symboling,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,3,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,3,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,1,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and continuous target `price`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['price'])
y = df['price']

print(f"Missing attributes: {X.isnull().sum().sum()}")


Missing attributes: 0


## 🔒 STEP 6: TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Vehicles: {X_train.shape[0]}")
print(f"Test Vehicles : {X_test.shape[0]}")


Train Vehicles: 164
Test Vehicles : 41


## ⚙️ STEP 7: HETEROGENEOUS COLUMN TRANSFORMER PREPROCESSOR
Encoding categorical vehicle attributes and standardizing continuous dimensions.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"Numerical features   ({len(num_cols)}): {num_cols}")


Categorical features (9): ['fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem']
Numerical features   (14): ['symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight', 'enginesize', 'boreratio', 'stroke', 'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg']


/tmp/ipykernel_682630/3601337051.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


## ⚔️ STEP 8: POLYNOMIAL VS RBF SVR DUEL
Benchmarking polynomial curvature against RBF radial basis kernels.


In [6]:
# STEP 8: KERNEL DUEL
pipe_poly = Pipeline([
    ('prep', preprocessor),
    ('svr', SVR(kernel='poly', degree=2, C=5000, epsilon=500))
])
pipe_poly.fit(X_train, y_train)

pipe_rbf = Pipeline([
    ('prep', preprocessor),
    ('svr', SVR(kernel='rbf', C=5000, epsilon=500))
])
pipe_rbf.fit(X_train, y_train)

r2_poly = r2_score(y_test, pipe_poly.predict(X_test))
r2_rbf = r2_score(y_test, pipe_rbf.predict(X_test))

print("="*65)
print(f"Polynomial (deg=2) SVR R²: {r2_poly*100:.2f}% | MAE: ${mean_absolute_error(y_test, pipe_poly.predict(X_test)):,.2f}")
print(f"RBF Gaussian SVR R²      : {r2_rbf*100:.2f}% | MAE: ${mean_absolute_error(y_test, pipe_rbf.predict(X_test)):,.2f}")
print("="*65)


Polynomial (deg=2) SVR R²: 85.93% | MAE: $2,199.95


RBF Gaussian SVR R²      : 80.10% | MAE: $2,480.02


/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown catego

## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing $C$, $\varepsilon$, and $\gamma$.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'svr__C': [2000, 5000, 10000],
    'svr__epsilon': [100, 500, 1000],
    'svr__gamma': ['scale', 0.05, 0.1]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_rbf, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Automotive Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²: {grid.best_score_*100:.2f}%")


/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unkno

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown cat

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown cat

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown cat

🏆 Best Automotive Hyperparameters: {'svr__C': 10000, 'svr__epsilon': 500, 'svr__gamma': 0.05}
🎯 Best 5-Fold CV R²: 84.68%


/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown catego

## 📊 STEP 10 & 11: VALUATION REPORT & RESIDUALS
Evaluating vehicle appraisal precision.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📋 AUTOMOTIVE VALUATION REPORT:")
print(f"   Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"   Root Mean Squared Error   : ${rmse:,.2f}")
print(f"   R² Score                  : {r2 * 100:.2f}%")


📋 AUTOMOTIVE VALUATION REPORT:
   Mean Absolute Error (MAE) : $2,094.56
   Root Mean Squared Error   : $3,256.01
   R² Score                  : 86.57%


/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & DEALERSHIP SMOKE TEST
Deploying the pipeline and validating instant trade-in appraisal latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/car_price_svr_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_car = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_price = loaded_pipe.predict(sample_car)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🚘 LIVE DEALERSHIP APPRAISAL AUDIT:")
print(f"   Estimated Market Value: 🏷️ ${pred_price:,.2f}")
print(f"   Actual Listed Price   : ${y_test.iloc[0]:,.2f}")
print(f"   Appraisal Latency     : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/car_price_svr_pipeline.joblib (11,190 bytes)

🚘 LIVE DEALERSHIP APPRAISAL AUDIT:
   Estimated Market Value: 🏷️ $25,754.39
   Actual Listed Price   : $30,760.00
   Appraisal Latency     : 14.268 ms (SLA < 2.0ms: CHECK)
